In [13]:
import pandas as pd
import os
from datetime import timedelta, datetime
import numpy as np

# --- Main Script Logic ---

def label_data():
    """
    Reads breakout data, processes stock files, and labels the data.
    Saves the labeled data to a new 'LabeledData' folder.
    """
    try:
        # Define paths
        breakout_file = "Breakout-consolidation-phase.csv"
        engineered_features_folder = "EngineeredFeatures"
        labeled_data_folder = "LabeledData"

        # Create the output folder if it doesn't exist
        os.makedirs(labeled_data_folder, exist_ok=True)

        # Read the breakout consolidation data
        breakout_df = pd.read_csv(breakout_file)
        print("Successfully read Breakout-consolidation-phase.csv")

        # --- CRITICAL FIX ---
        # Use a dictionary to store dataframes to avoid re-reading and overwriting labels.
        company_dataframes = {}

        # Iterate through each row in the breakout data
        for index, row in breakout_df.iterrows():
            company = row["Company"]
            date_str = row["Date"]

            # Try to get n_bars as a number, handling any potential errors.
            try:
                n_bars = int(row["N-bars"])
            except (ValueError, KeyError) as e:
                print(f"Skipping {company}: 'N-bars' value is missing or not a valid number. Error: {e}")
                continue

            # Check if we have already loaded this company's data
            if company not in company_dataframes:
                # Construct the path to the stock's data file, using the correct filename suffix.
                stock_file = os.path.join(engineered_features_folder, f"{company}-Engineered-Features.csv")

                # Check if the stock file exists
                if not os.path.exists(stock_file):
                    print(f"Skipping {company}: data file not found at {stock_file}")
                    continue

                # Read the stock's data, convert dates, and initialize the Label column.
                stock_df = pd.read_csv(stock_file)
                stock_df["Date"] = pd.to_datetime(stock_df["Date"], format="%Y-%m-%d", errors='coerce')
                stock_df["Label"] = "NotImportant" # Initialize the column once
                company_dataframes[company] = stock_df
                print(f"Processing data for {company}...")
            else:
                # If the company's data is already loaded, use it.
                stock_df = company_dataframes[company]
            
            # The dates in the breakout file are DD-MM-YYYY.
            breakout_date = pd.to_datetime(date_str, format="%d-%m-%Y", errors='coerce')

            # Find the index of the breakout week using the correctly parsed date.
            breakout_indices = stock_df[stock_df["Date"] == breakout_date].index
            
            if not breakout_indices.empty:
                breakout_index = breakout_indices[0]

                # Label the breakout week
                stock_df.loc[breakout_index, "Label"] = "Breakout"

                # Label the week before breakout as "EndOfConsolidation"
                if breakout_index > 0:
                    stock_df.loc[breakout_index - 1, "Label"] = "EndOfConsolidation"

                # Get a slice of the dataframe for the consolidation weeks
                # This labels the n_bars-2 weeks before the EndOfConsolidation week.
                consolidation_start_index = breakout_index - n_bars
                consolidation_end_index = breakout_index - 2
                
                if consolidation_start_index >= 0 and consolidation_end_index >= consolidation_start_index:
                    consolidation_rows = stock_df.loc[consolidation_start_index:consolidation_end_index]
                    
                    # Label the consolidation weeks using their robust indices.
                    stock_df.loc[consolidation_rows.index, "Label"] = "Consolidation"
                    
                    print(f"Labeled {len(consolidation_rows)} weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for {company}.")
                else:
                    print(f"Warning: Not enough data points to label a full consolidation period for {company}. Check n_bars value.")
            else:
                # Provide diagnostic information if the date is not found
                print(f"Warning: No date found in {company}'s data matching the breakout date of {breakout_date.date()}. Skipping labeling.")
                if not stock_df.empty:
                    print(f"Data range for {company}: from {stock_df['Date'].min().date()} to {stock_df['Date'].max().date()}")

        # --- NEW STEP ---
        # After the loop is complete, save all the labeled dataframes.
        for company, stock_df in company_dataframes.items():
            # Save the labeled dataframe to the new folder
            output_file_path = os.path.join(labeled_data_folder, f"{company}.csv")
            stock_df.to_csv(output_file_path, index=False)
            print(f"Saved labeled data for {company} to {output_file_path}")

    except FileNotFoundError as e:
        print(f"Error: The file {e.filename} was not found. Please ensure your CSV files are in the correct location.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the main function
if __name__ == "__main__":
    label_data()


Successfully read Breakout-consolidation-phase.csv
Processing data for HDFCBANK...
Labeled 5 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 4 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 4 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 19 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 4 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 13 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 3 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 9 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 29 weeks as 'Consolidation', 1 as 'EndOfConsolidation', and 1 as 'Breakout' for HDFCBANK.
Labeled 10 weeks as 'Consolidation', 1 as

In [14]:
import pandas as pd
import os
import numpy as np

# --- Main Script Logic ---

def treat_data():
    """
    Reads all labeled data files, combines them, removes the stock name,
    replaces empty fields with the column average, and saves to a new CSV file.
    """
    try:
        # Define paths
        labeled_data_folder = "LabeledData"
        output_file = "TreatedData.csv"
        
        # Check if the LabeledData folder exists
        if not os.path.exists(labeled_data_folder):
            print(f"Error: The folder '{labeled_data_folder}' was not found. Please run the labeling script first.")
            return

        # List to hold all dataframes
        all_dfs = []
        
        # Loop through all files in the LabeledData folder
        for filename in os.listdir(labeled_data_folder):
            if filename.endswith(".csv"):
                file_path = os.path.join(labeled_data_folder, filename)
                print(f"Reading {filename}...")
                
                # Read the CSV file into a dataframe
                df = pd.read_csv(file_path)
                all_dfs.append(df)

        if not all_dfs:
            print("No CSV files found in the LabeledData folder.")
            return

        # Concatenate all dataframes into a single one
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print("Successfully combined all labeled data into a single dataframe.")
        
        # Drop unnecessary columns. The 'Company' column is not needed as per the request,
        # and the first unnamed column is the old index from the original files.
        if 'Company' in combined_df.columns:
            combined_df = combined_df.drop('Company', axis=1)
            print("Removed the 'Company' column.")

        # --- Fill missing values ---
        # The goal is to replace NaNs with the average of the column.
        # This is a common method for handling missing data, known as mean imputation.
        
        # First, ensure that numeric columns are of the correct type.
        for col in combined_df.columns:
            # Skip the 'Label' and any other non-numeric columns
            if combined_df[col].dtype == 'object' and col != 'Label':
                # Attempt to convert to numeric, coercing errors to NaN
                combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
        
        # Now, calculate the average for each numeric column and fill missing values
        numeric_cols = combined_df.select_dtypes(include=np.number).columns
        print(f"Filling missing values for numeric columns: {list(numeric_cols)}")
        
        for col in numeric_cols:
            # Calculate the mean of the column, ignoring existing NaN values
            mean_value = combined_df[col].mean()
            # Fill the NaN values with the calculated mean
            combined_df[col] = combined_df[col].fillna(mean_value)
            
        print("Successfully replaced all empty fields with the column average.")

        # Save the final treated dataframe to a new CSV file
        combined_df.to_csv(output_file, index=False)
        print(f"Treated data saved to {output_file}")
        
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the main function
if __name__ == "__main__":
    treat_data()



Reading MM.csv...
Reading ICICIBANK.csv...
Reading MARUTI.csv...
Reading ASIANPAINT.csv...
Reading APOLLOHOSP.csv...
Reading HDFCBANK.csv...
Reading ADANIENT.csv...
Reading HEROMOTOCO.csv...
Reading SBIN.csv...
Reading BAJAJFINSV.csv...
Reading BAJFINANCE.csv...
Successfully combined all labeled data into a single dataframe.
Filling missing values for numeric columns: ['Date', 'Total Traded Quantity', 'Symbol', 'Weekly_Return', 'Normalized_Open', 'Normalized_High', 'Normalized_Low', 'EMA5_Diff_Pct', 'EMA13_Diff_Pct', 'EMA23_Diff_Pct', 'Volume_SMA10', 'Relative_Volume', 'Volume_Spike_Ratio_5Wk', 'Normalized_OBV', 'Normalized_Weekly_Range', 'ATR', 'Bollinger_Z_Score', 'ROC_10', 'RSI', 'Stochastic_%K', 'Stochastic_%D', 'VWAP_Diff_Pct']
Successfully replaced all empty fields with the column average.
Treated data saved to TreatedData.csv
